In [10]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

from imblearn.over_sampling import SMOTE


In [11]:
df = pd.read_csv('../Datasets/preprocessed_liver_data.csv')

X = df.drop('Result', axis=1)
y = df['Result']


In [12]:
skf = StratifiedKFold(
    n_splits=10,        
    shuffle=True,
    random_state=42
)


In [13]:
acc_scores = []
auc_scores = []
precision_scores = []
recall_scores = []
f1_scores = []


In [14]:
train_times = []
predict_times = []
fold = 1

for train_idx, val_idx in skf.split(X, y):
    print(f"\n--- Fold {fold} ---")
    
    # Split data
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Apply SMOTE ONLY on training fold
    smote = SMOTE(random_state=42)
    X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
    
    # Train Random Forest 
    rf = RandomForestClassifier(
        random_state=42,
        n_estimators=100
    )
    
    # Training time
    start_train = time.perf_counter()
    rf.fit(X_train_bal, y_train_bal)
    end_train = time.perf_counter()

    train_times.append(end_train - start_train)

    # Prediction time
    start_pred = time.perf_counter()
    y_pred = rf.predict(X_val)
    y_prob = rf.predict_proba(X_val)[:,1]
    end_pred = time.perf_counter()

    predict_times.append(end_pred - start_pred)
    
    # Predictions
    y_pred = rf.predict(X_val)
    y_pred_proba = rf.predict_proba(X_val)[:, 1]
    
    # Metrics
    acc = accuracy_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_pred_proba)
    prec = precision_score(y_val, y_pred)
    rec = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    
    # Store
    acc_scores.append(acc)
    auc_scores.append(auc)
    precision_scores.append(prec)
    recall_scores.append(rec)
    f1_scores.append(f1)
    
    print(f"Accuracy  : {acc:.4f}")
    print(f"ROC-AUC   : {auc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-score  : {f1:.4f}")
    
    fold += 1


--- Fold 1 ---
Accuracy  : 0.9964
ROC-AUC   : 0.9999
Precision : 0.9993
Recall    : 0.9957
F1-score  : 0.9975

--- Fold 2 ---
Accuracy  : 0.9959
ROC-AUC   : 1.0000
Precision : 0.9971
Recall    : 0.9971
F1-score  : 0.9971

--- Fold 3 ---
Accuracy  : 0.9959
ROC-AUC   : 0.9999
Precision : 0.9957
Recall    : 0.9986
F1-score  : 0.9971

--- Fold 4 ---
Accuracy  : 0.9943
ROC-AUC   : 0.9999
Precision : 0.9957
Recall    : 0.9964
F1-score  : 0.9960

--- Fold 5 ---
Accuracy  : 0.9954
ROC-AUC   : 0.9988
Precision : 0.9978
Recall    : 0.9957
F1-score  : 0.9967

--- Fold 6 ---
Accuracy  : 0.9990
ROC-AUC   : 1.0000
Precision : 0.9993
Recall    : 0.9993
F1-score  : 0.9993

--- Fold 7 ---
Accuracy  : 0.9954
ROC-AUC   : 0.9989
Precision : 0.9957
Recall    : 0.9978
F1-score  : 0.9967

--- Fold 8 ---
Accuracy  : 0.9948
ROC-AUC   : 0.9999
Precision : 0.9949
Recall    : 0.9978
F1-score  : 0.9964

--- Fold 9 ---
Accuracy  : 0.9964
ROC-AUC   : 1.0000
Precision : 0.9971
Recall    : 0.9978
F1-score  : 0.9975



In [15]:
print("\n====== Random Forest CV Results ======")

print(f"Accuracy  : {np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}")
print(f"ROC-AUC   : {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print(f"Precision : {np.mean(precision_scores):.4f}")
print(f"Recall    : {np.mean(recall_scores):.4f}")
print(f"F1-score  : {np.mean(f1_scores):.4f}")

print("\n====== Random Forest Timing ======")
    
print(f"Average Training Time  : {np.mean(train_times):.4f} seconds")
print(f"Average Prediction Time: {np.mean(predict_times):.6f} seconds")



====== Random Forest CV Results ======
Accuracy  : 0.9957 ± 0.0014
ROC-AUC   : 0.9996 ± 0.0005
Precision : 0.9970
Recall    : 0.9969
F1-score  : 0.9970

====== Random Forest Timing ======
Average Training Time  : 4.3589 seconds
Average Prediction Time: 0.104914 seconds
